# Kontextlängen-Ablation: SBERT vs. Long-Context Jina Embeddings v2

Dieses Notebook dokumentiert die systematische **Ablationsstudie zur Kontextlänge** bei der semantischen Bewertung von parallelen Dokumentenpaaren (Alltagssprache vs. Leichte/Einfache Sprache).

## 1. Wissenschaftliche Problemstellung
Klassische Sentence-Transformer-Modelle (wie `paraphrase-multilingual-MiniLM-L12-v2` oder `all-mpnet-base-v2`) besitzen eine harte Eingabelängenbegrenzung von **128 bis 512 Tokens**:
- **Truncation-Effekt**: Bei langen Fachtexten oder behördlichen Satzungen werden bis zu 95 % aller Dokumente vorzeitig abgeschnitten.
- **Introduction Bias in Leichter Sprache**: LS-Texte nutzen Einleitungsabsätze häufig für Lesehinweise, Begrüßungen oder Teaser-Boxen. Werden nur die ersten 128 Tokens encodiert, misst das Modell primär diese abweichende Rahmung statt der inhaltlichen Parallelität im Fließtext.

## 2. Lösung mit Long-Context Embeddings
Durch den Einsatz von `jinaai/jina-embeddings-v2-base-de` mit nativem **8.192-Token-Kontextfenster** (mittels ALiBi-Positionsgewichtung) werden 100 % aller Dokumente verlustfrei und ganzheitlich vektorisiert.

In [ ]:
import os, sys

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'data')) and os.path.exists(os.path.join(p, 'results')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser('~/Documents/Master Thesis'))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
import os
import sys
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm

while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir("..")

print(f"Arbeitsverzeichnis: {os.getcwd()}")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

## 3. Datenbasis & Coverage-Analyse
Wir analysieren die Token- und Wortlängenverteilung des bereinigten Master-Korpus (`data/analysis/corpus_master.json`).

In [ ]:
corpus_path = "data/analysis/corpus_master.json"
with open(corpus_path, "r", encoding="utf-8") as f:
    corpus_data = json.load(f)

df_corpus = pd.DataFrame(corpus_data)
print(f"Geladene Artikelpaare: {len(df_corpus)}")

# Token-Limits vergleichen
limits = [128, 256, 512, 1024, 2048, 8192]
coverage_rows = []

for lim in limits:
    as_covered = (df_corpus["as_tokens"] <= lim).mean() * 100
    ls_covered = (df_corpus["ls_tokens"] <= lim).mean() * 100
    coverage_rows.append({
        "Kontextlimit (Tokens)": lim,
        "AS Vollständig (%)": as_covered,
        "AS Abgeschnitten (%)": 100 - as_covered,
        "LS Vollständig (%)": ls_covered,
        "LS Abgeschnitten (%)": 100 - ls_covered,
    })

df_coverage = pd.DataFrame(coverage_rows)
display(df_coverage.set_index("Kontextlimit (Tokens)").round(1))

# Visualisierung der Coverage-Kurve
plt.figure(figsize=(10, 5))
plt.plot(df_coverage["Kontextlimit (Tokens)"], df_coverage["AS Vollständig (%)"], marker="o", label="Alltagssprache (AS)", color="#1f77b4", linewidth=2.5)
plt.plot(df_coverage["Kontextlimit (Tokens)"], df_coverage["LS Vollständig (%)"], marker="s", label="Leichte Sprache (LS)", color="#ff7f0e", linewidth=2.5)
plt.xscale("log", base=2)
plt.xticks(limits, [str(x) for x in limits])
plt.xlabel("Maximales Kontextfenster (Tokens, logarithmisch)")
plt.ylabel("Vollständig erfasste Artikel (%)")
plt.title("Coverage-Rate in Abhängigkeit des Kontextlimits")
plt.axvline(128, color="red", linestyle="--", alpha=0.7, label="Klassisches SBERT (128)")
plt.axvline(512, color="purple", linestyle="--", alpha=0.7, label="Standard Transformer (512)")
plt.axvline(8192, color="green", linestyle="--", alpha=0.7, label="Jina Embeddings v2 (8192)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 4. Evaluation der semantischen Ähnlichkeit (128 vs. 512 vs. 8.192 Tokens)
Wir führen die Vektorisierung für die verschiedenen Kontextlängen durch oder laden die vorberechnete CSV-Datei.

In [ ]:
ablation_csv = "data/analysis/jina_context_ablation.csv"

if os.path.exists(ablation_csv):
    print(f"Lade vorhandene Ablation-Ergebnisse aus: {ablation_csv}")
    df_ablation = pd.read_csv(ablation_csv)
else:
    print("Berechne Kontext-Ablation mit Jina Embeddings v2...")
    from sentence_transformers import SentenceTransformer, util
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "jinaai/jina-embeddings-v2-base-de"
    sbert = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    
    context_lengths = [128, 256, 512, 1024, 8192]
    results = []
    
    for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus), desc="Jina Inferenz"):
        as_text = str(row["as_text"]).strip()
        ls_text = str(row["ls_text"]).strip()
        
        item = {
            "source": row["source"],
            "as_url": row.get("as_url", ""),
            "ls_url": row.get("ls_url", ""),
            "as_tokens": row.get("as_tokens", len(as_text.split())),
            "ls_tokens": row.get("ls_tokens", len(ls_text.split()))
        }
        
        for ctx in context_lengths:
            sbert.max_seq_length = ctx
            emb_as = sbert.encode(as_text, convert_to_tensor=True, device=device)
            emb_ls = sbert.encode(ls_text, convert_to_tensor=True, device=device)
            item[f"sim_{ctx}"] = float(util.cos_sim(emb_as, emb_ls)[0][0].item())
            
        results.append(item)
        
    df_ablation = pd.DataFrame(results)
    os.makedirs(os.path.dirname(ablation_csv), exist_ok=True)
    df_ablation.to_csv(ablation_csv, index=False, encoding="utf-8")
    print(f"Ergebnisse gespeichert unter: {ablation_csv}")

## 5. Aggregierte Ergebnisse nach Quellen

In [ ]:
sim_cols = ["sim_128", "sim_256", "sim_512", "sim_1024", "sim_8192"]
available_cols = [c for c in sim_cols if c in df_ablation.columns]

summary_df = df_ablation.groupby("source")[available_cols].mean().reset_index()
if "sim_8192" in summary_df.columns and "sim_128" in summary_df.columns:
    summary_df["Delta (8192 - 128)"] = summary_df["sim_8192"] - summary_df["sim_128"]

# Spalten formatieren
rename_dict = {
    "source": "Quelle",
    "sim_128": "Jina (128)",
    "sim_256": "Jina (256)",
    "sim_512": "Jina (512)",
    "sim_1024": "Jina (1024)",
    "sim_8192": "Jina (8192 / Full)"
}
summary_formatted = summary_df.rename(columns=rename_dict)
display(summary_formatted.round(3).sort_values(by="Delta (8192 - 128)", ascending=False))

## 6. Visualisierungen 

In [ ]:
# Grouped Bar Chart (128 vs 512 vs 8192)
plot_cols = ["sim_128", "sim_512", "sim_8192"]
mean_sims = df_ablation.groupby("source")[plot_cols].mean().reset_index()
mean_sims = mean_sims.rename(columns={
    "sim_128": "128 Tokens (Klassisches SBERT)",
    "sim_512": "512 Tokens (Standard-Transformer)",
    "sim_8192": "8192 Tokens (Jina Full Context)"
})

sim_melted = mean_sims.melt(id_vars="source", var_name="Kontextlimit", value_name="Kosinus-Ähnlichkeit")

plt.figure(figsize=(14, 6))
palette = ["#d95f02", "#7570b3", "#1b9e77"]
ax = sns.barplot(data=sim_melted, x="source", y="Kosinus-Ähnlichkeit", hue="Kontextlimit", palette=palette)
plt.title("Semantische Ähnlichkeit: Einfluss der Kontextlänge (Jina Embeddings v2)", fontsize=14, weight="bold")
plt.xlabel("Quelle", fontsize=12)
plt.ylabel("Kosinus-Ähnlichkeit ($\\text{sim}_{\\cos}$)", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.ylim(0.4, 1.0)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.)
plt.tight_layout()

# Abbildung speichern
os.makedirs("research/img/analysis", exist_ok=True)
plt.savefig("research/img/analysis/jina_context_comparison.png", dpi=300)
plt.show()

In [ ]:
# Verteilungsanalyse: Boxplot über alle Quellen bei 8192 Tokens
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_ablation, x="source", y="sim_8192", color="#4C72B0", width=0.6, fliersize=3)
plt.axhline(0.60, color="red", linestyle="--", label="Sweet-Spot Untergrenze (0.60)")
plt.axhline(0.98, color="green", linestyle="--", label="Sweet-Spot Obergrenze (0.98)")
plt.title("Verteilung der semantischen Ähnlichkeit nach Quelle (Jina 8192 Tokens)", fontsize=14, weight="bold")
plt.xlabel("Quelle", fontsize=12)
plt.ylabel("Kosinus-Ähnlichkeit ($\\text{sim}_{\\cos}$)", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.ylim(0.4, 1.02)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()